Below is all about the analysis:

In [ ]:
import pandas as pd
PATH  = '<CLEANED_DATA_DIR>/trip_stop_events.csv'
trip_stop_events = pd.read_csv(PATH)

# ── Step 1: Drop rows with empty route_id (Missing Type B) ────
# route_id= Null means missing type B, with no route records, but only exist in stop record
before_drop = trip_stop_events.shape[0]
trip_stop_events = trip_stop_events.dropna(subset=['route_id'])
after_drop = trip_stop_events.shape[0]

print(f"Due to empty route_id, dropped {before_drop - after_drop} rows (Missing Type B)")
print(f"Current trip_stop_events shape : {trip_stop_events.shape}")


### Method 1:  Simple Linear Outlier Detection (rule_outlier_score)
We evaluate timing abnormality evaluating ONLY on `completed_diff` and `departed_diff`.
- **≤ 10 mins**: 100 points
- **10–20 mins**: Linear scaling down
- **≥ 20 mins**: 0 points (20 mins is approximately the 1.5 IQR upper bound)

Final score is the average of completed and departed scores.

In [ ]:
import numpy as np

# ── Step 2: Method 1 - Simple Linear Outlier Detection ────────
def calculate_linear_score(x):
    x = np.abs(x)
    return np.where(pd.isna(x), np.nan,
                    np.where(x <= 10, 100,
                             np.where(x < 20, 100 - 10 * (x - 10), 0)))

trip_stop_events['completed_score'] = calculate_linear_score(trip_stop_events['completed_diff'])
trip_stop_events['departed_score'] = calculate_linear_score(trip_stop_events['departed_diff'])

# Average the two scores (ignoring NaNs where possible)
trip_stop_events['rule_outlier_score'] = trip_stop_events[['completed_score', 'departed_score']].mean(axis=1)
pct_not_100 = (trip_stop_events["rule_outlier_score"].ne(100).sum() / trip_stop_events["rule_outlier_score"].notna().sum()) * 100

print("Method 1 (Final Score) logic complete.")
print(f"Percentage of rows with rule_outlier_score not equal to 100: {pct_not_100:.2f}%")
trip_stop_events[['completed_diff', 'completed_score', 'departed_diff', 'departed_score', 'rule_outlier_score']].head()


### Method 2: LOF Outlier Detection (LOF_outlier_score)
We focus exclusively on timing deviation signals (`abs_completed_diff` and `abs_departed_diff`) without winsorization, so that genuinely large time deviations are visible to the model. RobustScaler is applied before LOF. The raw LOF score is converted to a 0–100 normality scale (100 = fully normal, 0 = most anomalous).

In [ ]:
from sklearn.preprocessing import RobustScaler
from sklearn.neighbors import LocalOutlierFactor
import warnings
warnings.filterwarnings('ignore')

# ── Feature Engineering ───────────────────────────────────────
df_lof = trip_stop_events.copy()

# Boolean columns → float
bool_cols = ['completed', 'departed', 'is_anchor', 'is_school_x',
             'is_anchor_stop', 'is_pickup', 'is_dropoff', 'final_anchor']
for c in bool_cols:
    if c in df_lof.columns:
        df_lof[c] = df_lof[c].astype(float)

# Missingness indicators
df_lof['completed_not_happened']    = (df_lof['completed'] == 0.0).astype(float)
df_lof['completed_diff_real_missing'] = ((df_lof['completed'] == 1.0) & df_lof['completed_diff'].isna()).astype(float)
df_lof['departed_not_happened']     = (df_lof['departed'] == 0.0).astype(float)
df_lof['departed_diff_real_missing']  = ((df_lof['departed'] == 1.0) & df_lof['departed_diff'].isna()).astype(float)

# Median imputation for timing cols → absolute deviation
comp_med = df_lof['completed_diff'].median()
df_lof['completed_diff_filled'] = df_lof['completed_diff'].fillna(comp_med)
df_lof['abs_completed_diff']    = df_lof['completed_diff_filled'].abs()

dep_med = df_lof['departed_diff'].median()
df_lof['departed_diff_filled'] = df_lof['departed_diff'].fillna(dep_med)
df_lof['abs_departed_diff']    = df_lof['departed_diff_filled'].abs()

# Median imputation for non-timing continuous cols only
other_cont_cols = ['minutes_to_arrival', 'mileage', 'geofence_radius_m']
for c in other_cont_cols:
    if c in df_lof.columns:
        df_lof[c] = df_lof[c].fillna(df_lof[c].median())

# NOTE: No winsorization on abs_completed_diff / abs_departed_diff.
# We want genuinely large time deviations to remain visible to LOF.

print("LOF Feature engineering complete.")

# ── Timing-Only LOF ───────────────────────────────────────────
# Use only the two timing deviation features so LOF captures timing anomalies,
# not multivariate execution-pattern anomalies.
lof_features = ['abs_completed_diff', 'abs_departed_diff']
X = df_lof[lof_features].copy()

scaler = RobustScaler()
X_scaled = scaler.fit_transform(X)

lof = LocalOutlierFactor(n_neighbors=20, contamination=0.05)
outlier_preds = lof.fit_predict(X_scaled)

df_lof['outlier_flag'] = (outlier_preds == -1).astype(int)
df_lof['lof_score']    = -lof.negative_outlier_factor_

# Convert to 0–100 normality score (100 = normal, 0 = most anomalous)
lof_min = df_lof['lof_score'].min()
lof_max = df_lof['lof_score'].max()
if lof_max == lof_min:
    df_lof['LOF_outlier_score'] = 100.0
else:
    df_lof['LOF_outlier_score'] = (
        100 * (1 - (df_lof['lof_score'] - lof_min) / (lof_max - lof_min))
    ).round(2)

# Build df_final with key columns for downstream steps
keep_cols = [
    'trip_id', 'route_stop_id', 'LOF_outlier_score', 'outlier_flag',
    'abs_completed_diff', 'abs_departed_diff',
    'completed', 'departed',
    'completed_not_happened', 'departed_not_happened'
]
keep_cols = [c for c in keep_cols if c in df_lof.columns]
df_final = df_lof[keep_cols].copy()

print(f"Timing-Only LOF complete. Flagged {df_final['outlier_flag'].sum()} anomalies out of {len(df_final)} stops.")
df_final.sort_values('LOF_outlier_score', ascending=True).head(10)


In [ ]:
# ── Step 7: Compute Final Outlier Score ──────────────────────
# final_outlier_score = min(LOF_outlier_score, rule_outlier_score)
#
#   - LOF_outlier_score  : timing-based LOF (large time diffs → low score)
#   - rule_outlier_score : linear rule scoring (>=20 min diff → 0)
# If either method flags a stop as anomalous, the final score stays low.

import numpy as np

df_final = df_final.merge(
    df_lof[['trip_id', 'route_stop_id', 'rule_outlier_score']],
    on=['trip_id', 'route_stop_id'],
    how='left'
)

df_final['final_outlier_score'] = np.minimum(
    df_final['LOF_outlier_score'],
    df_final['rule_outlier_score']
).round(2)

# ── Reorder columns ───────────────────────────────────────────
priority_cols = ['trip_id', 'route_stop_id', 'rule_outlier_score',
                 'LOF_outlier_score', 'final_outlier_score', 'outlier_flag']
other_cols = [c for c in df_final.columns if c not in priority_cols]
df_final = df_final[priority_cols + other_cols]

print("Final outlier score computation complete.")
print(f"Score range : {df_final['final_outlier_score'].min():.2f} – {df_final['final_outlier_score'].max():.2f}")
print(f"Mean score  : {df_final['final_outlier_score'].mean():.2f}")

# ── Save Final Output ─────────────────────────────────────────
OUTPUT_PATH = '<CLEANED_DATA_DIR>/trip_stop_outliers.csv'
df_final.to_csv(OUTPUT_PATH, index=False)
print(f"\nSaved {len(df_final):,} rows to: {OUTPUT_PATH}")

# Show worst 10 stops
df_final[['trip_id', 'route_stop_id', 'rule_outlier_score', 'LOF_outlier_score', 'final_outlier_score', 'outlier_flag']]    .sort_values('final_outlier_score', ascending=True).head(10)


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd

BAR_COLOR = '#8B2020'

plot_df = df_final.copy()
plot_df['avg_time_diff'] = (plot_df['abs_completed_diff'] + plot_df['abs_departed_diff']) / 2

bins   = [0, 5, 10, 15, 20, 25, 30, np.inf]
labels = ['0-5', '5-10', '10-15', '15-20', '20-25', '25-30', '30+']
plot_df['time_diff_bin'] = pd.cut(plot_df['avg_time_diff'], bins=bins, labels=labels, right=False)

def annotate_bars(ax, fmt="{:.1f}%", suffix=""):
    for p in ax.patches:
        h = p.get_height()
        if pd.notna(h) and h > 0:
            ax.annotate(fmt.format(h) + suffix,
                        (p.get_x() + p.get_width() / 2., h),
                        ha='center', va='bottom', fontsize=10)

# ── Chart 1: Distribution of avg time diff (all stops) ───────
dist_df = (plot_df['time_diff_bin']
           .value_counts(normalize=True)
           .reset_index())
dist_df.columns = ['bin', 'percentage']
dist_df['percentage'] *= 100
dist_df = dist_df.sort_values('bin')

fig1, ax1 = plt.subplots(figsize=(9, 5))
sns.barplot(data=dist_df, x='bin', y='percentage', ax=ax1, color=BAR_COLOR)
ax1.set_title('Chart 1 — Distribution of Average Time Diff (All Stops)', fontsize=13)
ax1.set_xlabel('Average Time Difference (mins)')
ax1.set_ylabel('Percentage of Dataset (%)')
annotate_bars(ax1, fmt="{:.1f}%")
plt.tight_layout()
plt.show()

# ── Chart 2: Avg final_outlier_score by time diff (all stops) ─
score_df = (plot_df.groupby('time_diff_bin', observed=True)['final_outlier_score']
            .mean().dropna().reset_index())
score_df.columns = ['bin', 'mean_score']

fig2, ax2 = plt.subplots(figsize=(9, 5))
sns.barplot(data=score_df, x='bin', y='mean_score', ax=ax2, color=BAR_COLOR)
ax2.set_title('Chart 2 — Avg Final Outlier Score by Time Diff (All Stops)\n'
              '100 = fully normal  |  0 = most abnormal', fontsize=13)
ax2.set_xlabel('Average Time Difference (mins)')
ax2.set_ylabel('Average Final Outlier Score (0–100)')
ax2.set_ylim(0, 110)
annotate_bars(ax2, fmt="{:.1f}", suffix="")
plt.tight_layout()
plt.show()

